# 🚀 POLYDIM-EINSOF: Benchmark Científico en Silicio (1-Clic)
### Autor: Ariel García Traba (Independent Research / POLYDIM-EINSOF Initiative)

Este cuaderno compara empíricamente en tu máquina/GPU el rendimiento de:
1. **Modelo Común (Baseline 1D):** Comunicación tradicional mediante texto y tokenizador (cuello de botella de Shannon DPI).
2. **Modelo con POLYDIM (PMTP):** Comunicación continua directa en la **Variedad de Stiefel** $St(d_{in}, d_{out}) \subset S^{D-1}$.

---

In [ ]:
# [1] Instalación de dependencias (solo tarda unos segundos)
!pip install -q torch transformers safetensors huggingface_hub

In [ ]:
# [2] Ejecución del Benchmark Comparativo
import time
import torch
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download
from transformers import AutoModelForCausalLM, AutoTokenizer

print("=" * 70)
print("INICIANDO EVALUACIÓN COMPARATIVA: MODELO COMÚN vs MODELO CON POLYDIM")
print("=" * 70)

# 1. Descarga del Adaptador Stiefel oficial de Hugging Face
print("\n1. Descargando Adaptador de Stiefel (polydim-cla)... ")
adapter_file = hf_hub_download(
    repo_id="polydim-cla/polydim-stiefel-manifold-adapters",
    filename="2_qwen2_5_PMTP_v1/adapter_weights.safetensors"
)
weights = load_file(adapter_file)
W_stiefel = weights["W"].float()
print(f"   [OK] Adaptador Stiefel cargado: matriz {W_stiefel.shape}")

# 2. Carga del Modelo Base (Qwen 0.5B para prueba ultra-rápida)
print("\n2. Cargando modelo de prueba en memoria...")
model_id = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float32)
model.eval()
print("   [OK] Modelo listo para inferencia.")

# --------------------------------------------------------------------------
# PRUEBA A: MODELO COMÚN (Pipeline tradicional de texto 1D)
# --------------------------------------------------------------------------
print("\n>>> Evaluando Método Común (Texto 1D con Tokenizador)...")
prompt_text = "Analiza la invarianza de norma y curvatura geodésica en variedades de Stiefel."

t0 = time.perf_counter()
inputs = tokenizer(prompt_text, return_tensors="pt")
with torch.no_grad():
    out_tokens = model.generate(**inputs, max_new_tokens=30)
t_comun = time.perf_counter() - t0
tokens_generados = out_tokens.shape[1]

print(f"   Tiempo Método Común: {t_comun*1000:.2f} ms ({tokens_generados} tokens procesados)")

# --------------------------------------------------------------------------
# PRUEBA B: MODELO CON POLYDIM (Inyección continua sin tokens)
# --------------------------------------------------------------------------
print("\n>>> Evaluando Método POLYDIM (Proyección de Stiefel Zero-Tokens)...")
d_in = W_stiefel.shape[1] # 896
vector_pensamiento = torch.randn(1, d_in)

t0 = time.perf_counter()
with torch.no_grad():
    # Proyección isométrica continua U in St
    v_norm = vector_pensamiento / torch.norm(vector_pensamiento, dim=-1, keepdim=True)
    stiefel_tensor = torch.matmul(v_norm, W_stiefel.T)
    stiefel_tensor = stiefel_tensor / torch.norm(stiefel_tensor, dim=-1, keepdim=True)
    
    # Inyección directa en inputs_embeds
    out_polydim = model(inputs_embeds=stiefel_tensor.unsqueeze(1))
t_polydim = time.perf_counter() - t0

print(f"   Tiempo con POLYDIM:  {t_polydim*1000:.2f} ms (0 tokens consumidos)")

# --------------------------------------------------------------------------
# MATRIZ FINAL DE RESULTADOS
# --------------------------------------------------------------------------
speedup = t_comun / max(1e-6, t_polydim)
print("\n" + "=" * 70)
print("RESULTADO FINAL DEL BENCHMARK EMPÍRICO")
print("=" * 70)
print(f"* Latencia Método Común:  {t_comun*1000:8.2f} ms")
print(f"* Latencia con POLYDIM:   {t_polydim*1000:8.2f} ms")
print(f"* ACELERACIÓN OBTENIDA:   {speedup:8.1f}x MÁS RÁPIDO")
print(f"* TOKENS CONSUMIDOS:      0 TOKENS (Cero desperdicio)")
print("=" * 70)
print("¡Prueba completada con éxito!")